# Color Segmentation Pipeline — SAM + img2img

**Pipeline:**
1. Sample random color hint strokes from GT images
2. Use already-generated ControlNet images as segmentation input
3. SAM segments garment into regions
4. Assign user hint colors to regions
5. img2img adds realism

**Input:** ControlNet-generated images (already done) + GT images for hint sampling

**Output:** Realistically colored garment images

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'diffusers==0.21.4',
    'huggingface_hub==0.23.4',
    'transformers==4.38.2',
    'accelerate==0.27.2',
], check=True)

print('Done — restart kernel now before running any other cells')

In [1]:
# ── Install ────────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'diffusers==0.27.2',
    'transformers',
    'accelerate',
    'safetensors',
    'opencv-python',
    'scikit-image',
    'scipy',
], check=True)

# Install SAM
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'
], check=True)

# Download SAM checkpoint (ViT-B is fastest, good enough)
import os
sam_ckpt = '/kaggle/working/sam_vit_b.pth'
if not os.path.exists(sam_ckpt):
    subprocess.run([
        'wget', '-q', '-O', sam_ckpt,
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    ], check=True)
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already exists.')

SAM checkpoint already exists.


In [ ]:
import os, json, glob, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from pathlib import Path
from scipy.ndimage import gaussian_filter
from skimage.color import rgb2lab, lab2rgb

from diffusers import StableDiffusionImg2ImgPipeline
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

# ── CONFIG ─────────────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    'controlnet_images_dir': '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/generated',   # your generated images
    'gt_dir':                '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/ground_truth',
    'sketch_dir':            '/kaggle/input/datasets/humnafaisal1/controlnet-data/kaggle/working/controlnet_eval_outputs/sketches',
    'output_dir':            '/kaggle/working/sam_outputs',

    # SAM
    'sam_checkpoint':        '/kaggle/working/sam_vit_b.pth',
    'sam_model_type':        'vit_b',
    'sam_points_per_side':   16,
    'sam_min_mask_area':     800,      # ignore tiny regions
    'sam_iou_thresh':        0.88,
    'sam_stability_thresh':  0.92,

    # Color hint generation
    'n_strokes':             6,        # number of color strokes to sample
    'stroke_min_length':     30,
    'stroke_max_length':     100,
    'stroke_min_width':      4,
    'stroke_max_width':      12,

    # img2img
    'sd_model':              'runwayml/stable-diffusion-v1-5',
    'img2img_strength':      0.50,
    'guidance_scale':        7.5,
    'num_inference_steps':   30,
    'image_size':            512,

    'seed': 42,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Step 1 — Color Hint Generation

Sample random dragged strokes from GT images.
These simulate what a user would draw on the canvas.

In [ ]:
def sample_color_hints(gt_image_np, n_strokes=6,
                        min_length=30, max_length=100,
                        min_width=4,  max_width=12,
                        img_size=512):
    """
    Sample random dragged color strokes from GT image.
    Strokes are placed only on non-background (garment) pixels.

    Returns:
        color_map : (H,W,3) float32 [0,1] — RGB color at hint pixels
        hint_mask : (H,W,1) float32 [0,1] — 1 where hint exists
    """
    color_map = np.zeros((img_size, img_size, 3), dtype=np.float32)
    hint_mask = np.zeros((img_size, img_size, 1), dtype=np.float32)

    # Find non-background pixels (not near-white)
    non_bg = np.argwhere(np.any(gt_image_np < 0.80, axis=-1))
    if len(non_bg) < 20:
        return color_map, hint_mask   # blank/white image — skip

    n = np.random.randint(max(1, n_strokes-2), n_strokes+3)

    for _ in range(n):
        # Random start on garment
        start  = non_bg[np.random.choice(len(non_bg))]
        y0, x0 = int(start[0]), int(start[1])
        color  = gt_image_np[y0, x0].copy()

        # Random direction, length, width
        angle  = np.random.uniform(0, 2 * np.pi)
        length = np.random.randint(min_length, max_length)
        width  = np.random.randint(min_width, max_width)

        x1 = int(np.clip(x0 + length * np.cos(angle), 0, img_size-1))
        y1 = int(np.clip(y0 + length * np.sin(angle), 0, img_size-1))

        # Rasterize stroke
        n_pts = max(length * 2, 10)
        xs    = np.linspace(x0, x1, n_pts).astype(int)
        ys    = np.linspace(y0, y1, n_pts).astype(int)
        yy, xx = np.ogrid[:img_size, :img_size]
        half_w = width // 2

        for px, py in zip(xs, ys):
            mask_circle = (xx - px)**2 + (yy - py)**2 <= half_w**2
            color_map[mask_circle]    = color
            hint_mask[mask_circle, 0] = 1.0

    return color_map, hint_mask


def visualize_hints(gt_np, color_map, hint_mask, title='Color Hints'):
    """Show GT image alongside the sampled color hints."""
    hint_vis = gt_np.copy()
    hint_vis[hint_mask[:,:,0] > 0] = color_map[hint_mask[:,:,0] > 0]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(gt_np);   axes[0].set_title('GT Image');    axes[0].axis('off')
    axes[1].imshow(color_map); axes[1].set_title('Color Map'); axes[1].axis('off')
    axes[2].imshow(hint_vis); axes[2].set_title('Hints on GT'); axes[2].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ── Test on a few samples ──────────────────────────────────────────────────
gt_paths = sorted(Path(CONFIG['gt_dir']).glob('*.*'))[:5]
print(f'Testing hint generation on {len(gt_paths)} GT images...')

for gt_path in gt_paths[:3]:
    gt_pil = Image.open(gt_path).convert('RGB').resize(
        (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
    gt_np  = np.array(gt_pil, dtype=np.float32) / 255.0

    color_map, hint_mask = sample_color_hints(
        gt_np,
        n_strokes  = CONFIG['n_strokes'],
        min_length = CONFIG['stroke_min_length'],
        max_length = CONFIG['stroke_max_length'],
        min_width  = CONFIG['stroke_min_width'],
        max_width  = CONFIG['stroke_max_width'],
        img_size   = CONFIG['image_size'],
    )
    print(f'  {gt_path.name}: {int(hint_mask.sum())} hint pixels')
    visualize_hints(gt_np, color_map, hint_mask, title=gt_path.name)

## Step 2 — Load SAM and Segment ControlNet Images

In [ ]:
print('Loading SAM...')
sam = sam_model_registry[CONFIG['sam_model_type']](
    checkpoint=CONFIG['sam_checkpoint']
).to(device)

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side         = CONFIG['sam_points_per_side'],
    pred_iou_thresh         = CONFIG['sam_iou_thresh'],
    stability_score_thresh  = CONFIG['sam_stability_thresh'],
    min_mask_region_area    = CONFIG['sam_min_mask_area'],
)
print('SAM loaded.')


def segment_image(image_np_uint8):
    """
    Segment a ControlNet-generated image into regions using SAM.
    Returns list of masks sorted by area (largest first).

    image_np_uint8: (H,W,3) uint8
    """
    masks = mask_generator.generate(image_np_uint8)
    # Sort by area descending — largest = outermost garment region
    masks = sorted(masks, key=lambda m: m['area'], reverse=True)
    return masks


def filter_background_masks(masks, image_np, bg_thresh=0.85):
    """
    Remove masks that are mostly background (near-white pixels).
    Background regions don't need color assignment.
    """
    filtered = []
    img_float = image_np.astype(float) / 255.0
    for m in masks:
        seg      = m['segmentation']
        px       = img_float[seg]              # pixels in this mask
        mean_val = px.mean()                   # brightness
        # Skip if mostly white/bright — likely background
        if mean_val < bg_thresh:
            filtered.append(m)
    return filtered


def visualize_masks(image_np, masks, title='SAM Segmentation'):
    """Overlay colored masks on the image."""
    overlay = image_np.copy().astype(float) / 255.0
    colors  = plt.cm.tab20(np.linspace(0, 1, max(len(masks), 1)))

    for i, m in enumerate(masks):
        seg   = m['segmentation']
        color = colors[i % len(colors)][:3]
        overlay[seg] = 0.5 * overlay[seg] + 0.5 * np.array(color)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(image_np);  axes[0].set_title('ControlNet Output'); axes[0].axis('off')
    axes[1].imshow(overlay);   axes[1].set_title(f'SAM: {len(masks)} regions'); axes[1].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ── Test SAM on one ControlNet image ──────────────────────────────────────
cn_images = sorted(glob.glob(os.path.join(CONFIG['controlnet_images_dir'], '*.png')))
if len(cn_images) == 0:
    # Fallback — use val GT images as proxy for testing
    print('No ControlNet images found — using GT as proxy for SAM test')
    cn_images = [str(p) for p in sorted(Path(CONFIG['gt_dir']).glob('*.*'))[:5]]

print(f'Found {len(cn_images)} ControlNet images')

# Test on first image
test_img    = Image.open(cn_images[0]).convert('RGB').resize(
    (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
test_np     = np.array(test_img)
test_masks  = segment_image(test_np)
test_masks  = filter_background_masks(test_masks, test_np)
print(f'SAM found {len(test_masks)} garment regions (after background filter)')
visualize_masks(test_np, test_masks)

## Step 3 — Assign Colors to Regions

Match each segmented region to the nearest/most-covered color hint.
Build a colored region map for img2img input.

In [ ]:
def assign_colors_to_masks(masks, color_map, hint_mask, image_np):
    """
    Assign a color to each segmented region based on hint strokes.

    Strategy:
    - If a region contains hint pixels → use median of those hint colors
    - If no hints inside → find nearest hint point and use that color
    - If no hints at all → keep original ControlNet color for that region

    Returns:
        colored_np: (H,W,3) float32 [0,1] — flat colored regions
        region_colors: list of (mask, color) tuples for visualization
    """
    H, W       = hint_mask.shape[:2]
    colored_np = np.ones((H, W, 3), dtype=np.float32)  # white background
    img_float  = image_np.astype(float) / 255.0

    hint_points = np.argwhere(hint_mask[:,:,0] > 0.05)
    has_hints   = len(hint_points) > 0

    region_colors = []

    for m in masks:
        seg = m['segmentation']   # (H,W) boolean

        if has_hints:
            # Find hint pixels inside this region
            hints_inside = [(y, x) for y, x in hint_points if seg[y, x]]

            if hints_inside:
                # Use median of hint colors in this region
                colors_inside = np.array([color_map[y, x] for y, x in hints_inside])
                region_color  = np.median(colors_inside, axis=0)
            else:
                # Find nearest hint point to this region's centroid
                region_pts  = np.argwhere(seg)
                centroid    = region_pts.mean(axis=0)
                dists       = np.linalg.norm(
                    hint_points.astype(float) - centroid, axis=1)
                nearest_pt  = hint_points[np.argmin(dists)]
                region_color= color_map[nearest_pt[0], nearest_pt[1]]
        else:
            # No hints at all — keep ControlNet color
            region_color = img_float[seg].mean(axis=0)

        colored_np[seg] = region_color
        region_colors.append((seg, region_color))

    return colored_np, region_colors


def overlay_sketch_on_color(colored_np, sketch_np, line_threshold=0.5):
    """
    Draw sketch lines on top of colored regions.
    This gives img2img a structural signal to follow.

    colored_np : (H,W,3) float32 [0,1]
    sketch_np  : (H,W)   float32 [0,1] — 0=line, 1=white
    """
    result      = colored_np.copy()
    line_mask   = sketch_np < line_threshold
    result[line_mask] = 0.0   # black lines
    return result


def visualize_coloring(cn_np, colored_np, color_map, hint_mask, sketch_np=None):
    """Show the full coloring pipeline for one sample."""
    n_cols = 4 if sketch_np is not None else 3
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols*4, 4))

    axes[0].imshow(cn_np/255.0 if cn_np.max()>1 else cn_np)
    axes[0].set_title('ControlNet Output'); axes[0].axis('off')

    # Color hints visualized
    hint_vis = np.ones_like(colored_np)
    hint_vis[hint_mask[:,:,0]>0] = color_map[hint_mask[:,:,0]>0]
    axes[1].imshow(hint_vis)
    axes[1].set_title('Color Hints'); axes[1].axis('off')

    axes[2].imshow(colored_np)
    axes[2].set_title('Colored Regions'); axes[2].axis('off')

    if sketch_np is not None and n_cols == 4:
        final_input = overlay_sketch_on_color(colored_np, sketch_np)
        axes[3].imshow(final_input)
        axes[3].set_title('+ Sketch Lines (img2img input)'); axes[3].axis('off')

    plt.tight_layout()
    plt.show()


# ── Test color assignment ──────────────────────────────────────────────────
# Use matching GT for hint sampling
gt_paths_list = sorted(Path(CONFIG['gt_dir']).glob('*.*'))
sk_paths_list = sorted(Path(CONFIG['sketch_dir']).glob('*.*'))

gt_pil   = Image.open(gt_paths_list[0]).convert('RGB').resize(
    (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
gt_np    = np.array(gt_pil, dtype=np.float32) / 255.0
sk_pil   = Image.open(sk_paths_list[0]).convert('L').resize(
    (CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
sk_np    = np.array(sk_pil, dtype=np.float32) / 255.0

color_map, hint_mask = sample_color_hints(gt_np, n_strokes=CONFIG['n_strokes'])
colored_np, _        = assign_colors_to_masks(test_masks, color_map, hint_mask, test_np)

visualize_coloring(test_np, colored_np, color_map, hint_mask, sketch_np=sk_np)
print('Color assignment done.')

## Step 4 — LAB Blend

Take luminance (shading, texture) from ControlNet output.
Take hue/saturation from colored regions.
Result: correct structure + correct color, ready for img2img.

In [ ]:
def lab_blend(structured_np, colored_np, hint_mask, blend_sigma=20.0):
    """
    Blend structured image (luminance) with colored regions (hue/saturation).

    structured_np : (H,W,3) float32 [0,1] — ControlNet output
    colored_np    : (H,W,3) float32 [0,1] — flat colored regions
    hint_mask     : (H,W,1) float32 [0,1] — where user drew hints
    blend_sigma   : Gaussian spread for soft hint blending

    Returns blended (H,W,3) float32 [0,1]
    """
    struct_lab = rgb2lab(structured_np.clip(0,1))
    color_lab  = rgb2lab(colored_np.clip(0,1))

    blended_lab = struct_lab.copy()

    # Spread hint mask with Gaussian for smooth color transitions
    hint_weight = gaussian_filter(
        hint_mask[:,:,0].astype(float), sigma=blend_sigma)
    hint_weight = np.clip(hint_weight * 4.0, 0, 1)   # sharpen spread

    # Replace A and B (color) channels weighted by hint presence
    # Where hints exist: use colored_np hue
    # Where no hints: keep structured_np hue (neutral)
    blended_lab[:,:,1] = (hint_weight * color_lab[:,:,1] +
                          (1-hint_weight) * struct_lab[:,:,1])
    blended_lab[:,:,2] = (hint_weight * color_lab[:,:,2] +
                          (1-hint_weight) * struct_lab[:,:,2])

    blended_rgb = lab2rgb(blended_lab).astype(np.float32)
    return np.clip(blended_rgb, 0, 1)


# ── Test LAB blend ─────────────────────────────────────────────────────────
struct_float = test_np.astype(float) / 255.0
blended      = lab_blend(struct_float, colored_np, hint_mask)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(struct_float); axes[0].set_title('ControlNet (structure)'); axes[0].axis('off')
axes[1].imshow(colored_np);   axes[1].set_title('Colored regions (flat)'); axes[1].axis('off')
axes[2].imshow(blended);      axes[2].set_title('LAB blend (structure+color)'); axes[2].axis('off')
plt.tight_layout()
plt.show()
print('LAB blend done.')

## Step 5 — img2img

Takes LAB-blended image as starting point.
Adds realism, texture, cleans background.
Strength=0.5 — preserves structure and color, adds fabric detail.

In [ ]:
print('Loading img2img pipeline...')
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    CONFIG['sd_model'],
    torch_dtype    = torch.float16,
    safety_checker = None,
).to(device)
print('img2img pipeline loaded.')


def rgb_to_color_name(rgb_01):
    """Convert RGB [0,1] to a basic color name for the prompt."""
    import colorsys
    r, g, b = rgb_01
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    h = h * 360
    if v < 0.2:                       return 'black'
    if v > 0.85 and s < 0.12:         return 'white'
    if s < 0.12:                       return 'grey'
    if h < 15 or h >= 345:            return 'red'
    if 15  <= h < 40:                  return 'orange'
    if 40  <= h < 70:                  return 'yellow'
    if 70  <= h < 155:                 return 'green'
    if 155 <= h < 200:                 return 'teal'
    if 200 <= h < 260:                 return 'blue'
    if 260 <= h < 290:                 return 'purple'
    if 290 <= h < 345:                 return 'pink'
    return 'colourful'


def extract_dominant_colors(color_map, hint_mask, n_colors=2):
    """Extract dominant colors from hint strokes using KMeans."""
    from sklearn.cluster import KMeans
    mask    = hint_mask[:,:,0] > 0.05
    pixels  = color_map[mask]
    if len(pixels) < 10:
        return []
    n       = min(n_colors, len(pixels))
    km      = KMeans(n_clusters=n, n_init=3, random_state=42).fit(pixels)
    counts  = np.bincount(km.labels_)
    centers = km.cluster_centers_[np.argsort(-counts)]
    names, seen = [], set()
    for c in centers:
        name = rgb_to_color_name(c)
        if name not in seen:
            names.append(name)
            seen.add(name)
    return names


def build_prompt(color_names, category='clothing garment'):
    base = 'white background, studio product photo, high quality, flat lay'
    if not color_names:
        return f'a {category}, {base}'
    if len(color_names) == 1:
        return f'a {color_names[0]} {category}, {base}'
    color_str = ' and '.join(color_names)
    return f'a {color_str} {category}, {base}'


@torch.no_grad()
def run_img2img(blended_np, color_map, hint_mask, category='clothing garment',
                strength=0.50, guidance_scale=7.5, n_steps=30, seed=42):
    """
    Run img2img on the LAB-blended image.
    Returns PIL output image.
    """
    input_pil = Image.fromarray(
        (blended_np * 255).astype(np.uint8)
    ).resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)

    color_names  = extract_dominant_colors(color_map, hint_mask)
    prompt       = build_prompt(color_names, category)
    neg_prompt   = ('blurry, low quality, deformed, person, mannequin, '
                    'grey background, dark background, extra garments, clutter')

    print(f'  Prompt: {prompt}')

    result = img2img_pipe(
        prompt          = prompt,
        negative_prompt = neg_prompt,
        image           = input_pil,
        strength        = strength,
        guidance_scale  = guidance_scale,
        num_inference_steps = n_steps,
        generator       = torch.Generator(device).manual_seed(seed),
    ).images[0]

    return result


print('img2img functions ready.')

## Step 6 — Full Pipeline Test

Runs everything end-to-end on a few samples.
Shows: Sketch | Color Hints | ControlNet | Blended | Final | GT

In [ ]:
def run_full_pipeline(cn_image_path, gt_path, sketch_path,
                       category='clothing garment', n_test_strengths=1):
    """
    Full pipeline for one sample:
    1. Load ControlNet image, GT, sketch
    2. Sample color hints from GT
    3. SAM segmentation on ControlNet image
    4. Assign colors to regions
    5. LAB blend
    6. img2img
    7. Visualize all stages
    """
    size = CONFIG['image_size']

    # ── Load images ──
    cn_pil  = Image.open(cn_image_path).convert('RGB').resize((size,size), Image.BILINEAR)
    gt_pil  = Image.open(gt_path).convert('RGB').resize((size,size), Image.BILINEAR)
    sk_pil  = Image.open(sketch_path).convert('L').resize((size,size), Image.BILINEAR)

    cn_np   = np.array(cn_pil)                             # uint8
    gt_np   = np.array(gt_pil, dtype=np.float32) / 255.0
    sk_np   = np.array(sk_pil, dtype=np.float32) / 255.0

    # ── 1. Sample color hints from GT ──
    color_map, hint_mask = sample_color_hints(
        gt_np,
        n_strokes  = CONFIG['n_strokes'],
        min_length = CONFIG['stroke_min_length'],
        max_length = CONFIG['stroke_max_length'],
        min_width  = CONFIG['stroke_min_width'],
        max_width  = CONFIG['stroke_max_width'],
        img_size   = size,
    )
    print(f'  Hint pixels: {int(hint_mask.sum())}')

    # ── 2. SAM segmentation ──
    masks = segment_image(cn_np)
    masks = filter_background_masks(masks, cn_np)
    print(f'  SAM regions: {len(masks)}')

    if len(masks) == 0:
        print('  WARNING: No garment regions found — using Gaussian spread fallback')
        # Fallback: Gaussian spread directly on blended image
        colored_np = np.ones((size,size,3), dtype=np.float32) * 0.85
        for c in range(3):
            spread = gaussian_filter(color_map[:,:,c] * hint_mask[:,:,0], sigma=40)
            weight = gaussian_filter(hint_mask[:,:,0].astype(float), sigma=40).clip(1e-6)
            colored_np[:,:,c] = (spread / weight).clip(0,1)
    else:
        # ── 3. Assign colors to regions ──
        colored_np, _ = assign_colors_to_masks(masks, color_map, hint_mask, cn_np)

    # ── 4. LAB blend ──
    cn_float = cn_np.astype(float) / 255.0
    blended  = lab_blend(cn_float, colored_np, hint_mask)

    # ── 5. img2img ──
    strengths = [CONFIG['img2img_strength']]
    results   = []
    for s in strengths:
        print(f'  Running img2img strength={s}...')
        out = run_img2img(
            blended, color_map, hint_mask,
            category       = category,
            strength       = s,
            guidance_scale = CONFIG['guidance_scale'],
            n_steps        = CONFIG['num_inference_steps'],
            seed           = CONFIG['seed'],
        )
        results.append((s, out))

    # ── 6. Visualize ──
    n_cols = 5 + len(results)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols*3.5, 4))

    # Hint visualization
    hint_vis = np.ones((size,size,3), dtype=np.float32)
    hint_vis[hint_mask[:,:,0]>0] = color_map[hint_mask[:,:,0]>0]

    panels = [
        (sk_np,      'Sketch',            'gray'),
        (hint_vis,   'Color Hints',       None),
        (cn_float,   'ControlNet',        None),
        (colored_np, 'Colored Regions',   None),
        (blended,    'LAB Blend',         None),
    ]
    for s, out in results:
        panels.append((np.array(out)/255.0, f'img2img s={s}', None))
    panels.append((gt_np, 'Ground Truth', None))

    for i, (img, title, cmap) in enumerate(panels[:len(axes)]):
        if cmap:
            axes[i].imshow(img, cmap=cmap)
        else:
            axes[i].imshow(img.clip(0,1))
        axes[i].set_title(title, fontsize=9)
        axes[i].axis('off')

    plt.suptitle(Path(cn_image_path).stem, fontsize=11)
    plt.tight_layout()

    out_path = os.path.join(CONFIG['output_dir'],
                            f'{Path(cn_image_path).stem}_pipeline.png')
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved: {out_path}')

    return results[0][1]   # return final image


print('Pipeline function ready.')

In [ ]:
# ── Run on N samples ────────────────────────────────────────────────────────
N_SAMPLES = 5   # adjust as needed

gt_paths_all  = sorted(Path(CONFIG['gt_dir']).glob('*.*'))
sk_paths_all  = sorted(Path(CONFIG['sketch_dir']).glob('*.*'))
cn_paths_all  = sorted(glob.glob(os.path.join(CONFIG['controlnet_images_dir'], '*.png')))

# If ControlNet images are named differently, pair by index
n_available = min(N_SAMPLES, len(gt_paths_all), len(sk_paths_all))
if len(cn_paths_all) == 0:
    print('No ControlNet images found — using GT as ControlNet proxy')
    cn_paths_all = [str(p) for p in gt_paths_all]

indices = random.sample(range(min(len(gt_paths_all), len(cn_paths_all))), n_available)

print(f'Running pipeline on {n_available} samples...')
for i, idx in enumerate(indices):
    print(f'\nSample {i+1}/{n_available} (index {idx})')
    cn_path = cn_paths_all[idx % len(cn_paths_all)]
    gt_path = str(gt_paths_all[idx])
    sk_path = str(sk_paths_all[idx])

    try:
        run_full_pipeline(
            cn_image_path = cn_path,
            gt_path       = gt_path,
            sketch_path   = sk_path,
            category      = 'clothing garment',
        )
    except Exception as e:
        print(f'  Error on sample {idx}: {e}')
        import traceback; traceback.print_exc()
        continue

print('\nAll samples done.')

## Step 7 — Strength Comparison

Test different img2img strength values on one sample to find the best tradeoff
between structure preservation and realism added.

In [ ]:
# Pick one sample for strength comparison
test_idx    = indices[0]
cn_path     = cn_paths_all[test_idx % len(cn_paths_all)]
gt_path     = str(gt_paths_all[test_idx])
sk_path     = str(sk_paths_all[test_idx])
size        = CONFIG['image_size']

cn_pil  = Image.open(cn_path).convert('RGB').resize((size,size), Image.BILINEAR)
gt_pil  = Image.open(gt_path).convert('RGB').resize((size,size), Image.BILINEAR)
sk_pil  = Image.open(sk_path).convert('L').resize((size,size), Image.BILINEAR)
cn_np   = np.array(cn_pil)
gt_np   = np.array(gt_pil, dtype=np.float32) / 255.0

color_map, hint_mask = sample_color_hints(gt_np, n_strokes=CONFIG['n_strokes'])
masks    = filter_background_masks(segment_image(cn_np), cn_np)
colored_np, _ = assign_colors_to_masks(masks, color_map, hint_mask, cn_np)
blended  = lab_blend(cn_np.astype(float)/255.0, colored_np, hint_mask)

strengths = [0.35, 0.45, 0.55, 0.65]
outputs   = []
for s in strengths:
    print(f'Testing strength={s}...')
    out = run_img2img(blended, color_map, hint_mask, strength=s,
                      n_steps=CONFIG['num_inference_steps'],
                      seed=CONFIG['seed'])
    outputs.append((s, out))

fig, axes = plt.subplots(1, len(strengths)+2, figsize=((len(strengths)+2)*4, 4))
axes[0].imshow(blended.clip(0,1)); axes[0].set_title('LAB Blend (input)'); axes[0].axis('off')
for i, (s, out) in enumerate(outputs):
    axes[i+1].imshow(np.array(out)/255.0)
    axes[i+1].set_title(f'strength={s}'); axes[i+1].axis('off')
axes[-1].imshow(gt_np); axes[-1].set_title('Ground Truth'); axes[-1].axis('off')
plt.suptitle('Strength Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'strength_comparison.png'), dpi=100)
plt.show()
print('Strength comparison saved.')